# ⚽ Futbol Bahis Analiz ve Gelişmiş Tahmin Uygulaması

Bu Jupyter Notebook çalışmasında, Sporx web sitesinden takımların fikstür ve maç sonuçlarını anlık olarak çeken (**Web Scraping**), bu verilerle iki takım arasındaki karşılaşmalar için **gelişmiş olasılık modelleri (Poisson Dağılımı)** kullanan ve kullanıcıya görsel bir ekran (**Tkinter GUI**) sunan modern bir masaüstü yazılımının yapımını öğreneceğiz.

Bu notebook, projeyi inceleyen kişinin hem **kullanılan kütüphaneleri sıfırdan öğrenmesini**, hem **arkadaki matematiksel algoritmayı kavramasını**, hem de **kodun satır satır mantığını çözmesini** amaçlar. Aceleye getirilmeden, en detaylı açıklamalar ve görsel/matematiksel modellemelerle hazırlanmıştır.

### 🧠 Neler Öğreneceksiniz?
1. **Python Kütüphaneleri**: `requests`, `BeautifulSoup` (`bs4`), `tkinter`, `math`, `datetime` kütüphanelerinin ne işe yaradığı ve neden kullanıldığı.
2. **Web Scraping & DOM Ağacı**: HTML iskeleti, etiketler (`<tr>`, `<td>`, `<div>`, `<a>`), sınıf (`class`) seçicileri ve `User-Agent` başlığıyla tarayıcı taklidi yapma.
3. **Türkçe Karakter Kodlama Hatası (Unicode)**: Python'un Türkçe karakterleri (özellikle `İ` ve `ı` harflerini) küçültürken yaptığı Unicode birleştirme karakterleri hatasından (`\u0307` gibi) kaçınarak güvenli URL oluşturma.
4. **Poisson Olasılık Modeli**: Futbolda gol tahmini yapmanın matematiksel modeli olan Poisson formülü, ortalama gol beklentisi ($\lambda$) ve iki takımın skor matrisi (Score Grid).
5. **Gelişmiş Form Decay (Zaman Ağırlıklandırması)**: Son oynanan maçlara en yüksek ağırlığı verip ($w = 0.85^i$) eski maçların etkisini üstel olarak azaltan algoritma.
6. **Saha Avantajı (Home/Away Factor)**: Takımların iç saha ve dış saha performanslarını %60/%40 oranlarıyla kombine etme.
7. **Tarih Kontrolü ve Derbi Tespiti**: Karşılaşan takımların 1 hafta (7 gün) içinde maçları olup olmadığını kontrol eden tarih ayrıştırıcı ve fark hesaplayıcı.
8. **Tkinter Arayüz Tasarımı**: Grafiksel kullanıcı arayüzü pencereleri, metin giriş kutuları, grid yerleşim planı ve olay döngüsü (Event Loop).

---
**NOT:** Hücreleri sırayla çalıştırmak için hücreyi seçip **Shift + Enter** tuş kombinasyonunu kullanabilirsiniz.

## 1. Bölüm: Kullanılan Kütüphaneler ve Seçilme Nedenleri

Python'da sıfırdan her şeyi yazmak yerine, belirli işlerde uzmanlaşmış standart ve üçüncü parti kütüphaneleri kullanırız. Projemizde yer alan kütüphanelerin detaylı açıklamaları:

* **`requests`**:
  * **Nedir?** Python'un internet üzerindeki web sunucularıyla haberleşmesini sağlayan en popüler kütüphanedir.
  * **Amacı**: Belirlediğimiz Sporx URL adresine HTTP GET isteği gönderip, sayfanın ham HTML kodlarını bir metin (string) olarak belleğe indirir.
  * **Önemli Ayrıntı**: `requests.get(url, timeout=10)` fonksiyonunda `timeout` (zaman aşımı) parametresi kullanırız. Eğer internet yavaşsa veya sunucu çökmüşse, programın sonsuza kadar yanıt vermez halde kalmasını önler, 10 saniye sonra hata fırlatır.

* **`BeautifulSoup` (`bs4` kütüphanesinden)**:
  * **Nedir?** İndirilen ham ve karmaşık HTML kodlarını parse eden (ayrıştıran) bir kütüphanedir.
  * **Amacı**: HTML kodlarını hiyerarşik bir ağaç yapısına çevirerek bizim etiket adına (`tr`, `td`, `a` vb.) veya CSS sınıflarına (`class_="score"` gibi) göre arama yapabilmemizi sağlar.

* **`tkinter` ve `ttk`**:
  * **Nedir?** Python'un varsayılan Grafiksel Kullanıcı Arayüzü (GUI) kütüphanesidir.
  * **Amacı**: Kullanıcıya bir pencere sunar, takımları girebilmesi için metin kutuları (`Entry`) hazırlar ve "Analiz Yap" butonuna tıklandığında kodları çalıştırır.
  * **Farkı**: Klasik `tkinter` nesneleri daha eski bir görünüme sahipken, `ttk` (Themed Tkinter) modern işletim sistemi temasıyla uyumlu görsel bileşenler sunar.

* **`math`**:
  * **Nedir?** Python'un yerleşik matematik fonksiyonları kütüphanesidir.
  * **Amacı**: Poisson olasılık formülündeki üslü Euler sayısı ($e^{-\lambda}$) için `math.exp` ve faktöriyel ($k!$) hesaplaması için `math.factorial` fonksiyonlarını sağlar.

* **`datetime`**:
  * **Nedir?** Tarih ve zaman işlemlerini gerçekleştiren standart modüldür.
  * **Amacı**: Web sitesinden çekilen metin formatındaki maç tarihlerini gerçek tarih nesnelerine (`datetime.date`) çevirir ve bugünün tarihiyle karşılaştırarak aradaki gün farkını ölçer.

* **`os`**:
  * **Nedir?** İşletim sistemiyle doğrudan iletişim kuran modüldür.
  * **Amacı**: Analiz yenilendiğinde konsol ekranını Windows için `cls`, macOS/Linux için `clear` komutuyla temiz tutar.

In [ ]:
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import requests
from bs4 import BeautifulSoup
import os
import math
import datetime

## 2. Bölüm: Web Scraping ve HTML Yapısının Çözülmesi

Bir web tarayıcısında (Chrome, Edge vb.) bir siteyi açtığınızda arka planda bir HTML kodu çalışır. Web Scraping, bu kodu program aracılığıyla tarayarak içerisinden sadece ihtiyacımız olan kısımları çekme işlemidir.

### 2.1 HTML DOM (Document Object Model) Yapısı
HTML kodları etiketlerle (tag) yazılır ve bu etiketler ağaç hiyerarşisi oluşturur. Sporx sitesinin fikstür tablosunun yapısını inceleyelim:

* **`<tr>` (Table Row)**: Tablodaki her bir yatay satırı temsil eder. Fikstür sayfasındaki her bir futbol maçı bir `<tr>` satırıdır.
* **`<div class="team home">`**: Ev sahibi takımın adını ve logosunu barındıran kutudur. Altında yer alan `<a class="team-name">` etiketi ise takım ismini taşır.
* **`<a class="score">`**: Oynanan maçın skorunu (örn: `2-1`) taşıyan link etiketidir. Eğer maç henüz oynanmadıysa bu etiket içeriği `"-"` şeklinde görünür.
* **`<div class="team away">`**: Deplasman takımının bilgilerini barındıran kutudur.

BeautifulSoup ağaç yapısında gezinirken `find_all("tr")` diyerek tüm satırları çekeriz. Sonrasında her satırın içindeki skor ve takım div'lerini inceleyerek verileri temizleriz.

### 2.2 Tarayıcı Taklidi (User-Agent) Neden Önemlidir?
Web siteleri, aşırı trafik yükünü ve veri çekmeye çalışan botları engellemek için güvenlik duvarları kullanır. Python ile bir siteye istek gönderildiğinde, HTTP isteğinin başlığındaki `User-Agent` bilgisi varsayılan olarak `Python-requests/2.X` şeklinde gider. Sunucu bot olduğunu anlayıp `403 Forbidden` (Erişim Engellendi) hatası döndürür.

Bu engeli aşmak için istek gönderirken sunucuya gerçek bir tarayıcı bilgisi göndermemiz gerekir. Kodumuzda kullandığımız `headers` parametresi bu işi üstlenir:

```python
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 ..."
}
```

### 2.3 Türkçe Karakter ve Unicode Hataları
Sitedeki takım sayfaları SEO uyumlu URL kullanır. Örneğin: `fenerbahce-fiksturu-ve-mac-sonuclari`. Kullanıcı ise programa Türkçe harflerle `Fenerbahçe` yazabilir. Bu harfleri değiştirmemiz gerekir.

**Çok Önemli Detay (Büyük İ Hatası):** Python dilinde Türkçe karakter setinde `İ` harfi küçük harfe çevrilirken bazı platformlarda Unicode birleştirme karakteri olan `i\u0307` haline gelebilir. Bu durum sunucuya giden URL'yi bozar ve `404 Not Found` hatasına almamıza sebep olur. Bu yüzden öncelikle büyük `İ` ve `I` harflerini manuel olarak küçük `i` harfine çevirdikten sonra genel `.lower()` metodunu çağırırız.

In [ ]:
def clear_screen():
    '''Terminal/Konsol ekranını işletim sistemine uygun komutla temizler.'''
    os.system('cls' if os.name == 'nt' else 'clear')

def turkce_karakter_degistir(takim_ad):
    '''
    Kullanıcının girdiği takım adındaki Türkçe karakterleri temizler ve
    küçük harfe çevirerek Sporx URL'sine uygun (SEO uyumlu) hale getirir.
    '''
    # Unicode birleştirme karakterleri hatasını engellemek için büyük İ ve I'yı güvenle küçük i yaparız
    takim_ad = takim_ad.replace("İ", "i").replace("I", "i")
    
    # Tüm metni küçük harfe çeviririz
    takim_ad = takim_ad.lower()
    
    # Türkçe küçük harfleri İngilizce karşılıklarıyla değiştiririz
    takim_ad = takim_ad.replace("ı", "i")
    takim_ad = takim_ad.replace("ç", "c")
    takim_ad = takim_ad.replace("ş", "s")
    takim_ad = takim_ad.replace("ğ", "g")
    takim_ad = takim_ad.replace("ü", "u")
    takim_ad = takim_ad.replace("ö", "o")
    
    # Kelimeler arasındaki boşlukları tireye (-) dönüştürüp döndürürüz
    return takim_ad.replace(" ", "-")

# Algoritmayı test edelim
print("Girdi: Beşiktaş JK    -> Çıktı:", turkce_karakter_degistir("Beşiktaş JK"))
print("Girdi: İSTANBULSPOR   -> Çıktı:", turkce_karakter_degistir("İSTANBULSPOR"))
print("Girdi: Başakşehir     -> Çıktı:", turkce_karakter_degistir("Başakşehir"))

## 3. Bölüm: Poisson Dağılımı ve Olasılık Teorisinin Futbola Uygulanması

Bahis ve maç tahminleri yaparken rastgele tahminlerde bulunmak yerine olasılık teorisini kullanırız. Futbol gibi düşük skorlu ve bağımsız sayılabilecek olayların gerçekleştiği sporlarda en çok tercih edilen matematiksel model **Poisson Olasılık Dağılımı**'dır.

### 3.1 Poisson Dağılımı Nedir?
Poisson dağılımı, belirli bir zaman aralığında (örn: 90 dakikalık futbol maçında), sabit bir ortalamaya (beklenen ortalama gol sayısı) sahip bağımsız olayların (gollerin) gerçekleşme olasılıklarını hesaplar.

Poisson modelinin futbola uygulanabilmesi için temel varsayımlar:
1. **Sabit Ortalama**: Takımların gol atma ve yeme kapasitelerinin maç boyunca sabit olduğu varsayılır.
2. **Bağımsızlık**: Maçın 10. dakikasında atılan bir golün, 50. dakikasında atılacak başka bir golün olasılığını etkilemediği varsayılır (Modeli basitleştirmek için kabul edilen standart varsayımdır).
3. **Aynı Anda Gerçekleşmeme**: Aynı milisaniyede iki golün birden atılamayacağı kabul edilir.

### 3.2 Matematiksel Poisson Formülü
Bir takımın bir maçta tam olarak $k$ adet gol atma olasılığı $P(k; \lambda)$ şu formülle bulunur:

$$P(k; \lambda) = \frac{\lambda^k \cdot e^{-\lambda}}{k!}$$

Burada:
* **$\lambda$ (Lambda)**: Takımın o maçta atması beklenen **ortalama gol sayısıdır**.
* **$k$**: Hesaplanmak istenen gol hedefidir ($0, 1, 2, 3, \dots$).
* **$e$**: Euler sayısıdır (yaklaşık $2.71828$).
* **$k!$**: $k$ sayısının faktöriyelidir ($3! = 3 \times 2 \times 1 = 6$).

### 3.3 Skor Matrisi (Score Grid) ve Bağımsız Olasılıklar
Eğer Ev Sahibi takımın beklenen golü $\lambda_{ev}$ ve Deplasman takımının beklenen golü $\lambda_{dep}$ ise, maçın tam olarak $x - y$ skoruyla bitme olasılığı, her iki takımın gol atma olasılıklarının çarpımına eşittir:

$$P(\text{Skor} = x-y) = P(x; \lambda_{ev}) \times P(y; \lambda_{dep})$$

Örneğin, maçın **2-1** bitme ihtimali:
$$P(2-1) = \left( \frac{\lambda_{ev}^2 \cdot e^{-\lambda_{ev}}}{2!} \right) \times \left( \frac{\lambda_{dep}^1 \cdot e^{-\lambda_{dep}}}{1!} \right)$$

### 3.4 Alt / Üst 2.5 Olasılık Hesabı
Bahis oyunlarında en popüler seçeneklerden biri olan **Alt/Üst 2.5 Gol** tahminini hesaplamak için olasılık toplamını kullanırız. Maçta toplam gol sayısının en fazla 2 olması (**Alt 2.5**) durumu şu 6 skor kombinasyonuyla gerçekleşebilir:
* 0-0, 1-0, 0-1, 2-0, 1-1, 0-2

Bu 6 skorun olasılıklarını toplarız:
$$P(\text{Alt 2.5}) = P(0-0) + P(1-0) + P(0-1) + P(2-0) + P(1-1) + P(0-2)$$

Bir maç ya Alt ya da Üst biteceğinden, **Üst 2.5** gol olasılığı:
$$P(\text{Üst 2.5}) = 1.0 - P(\text{Alt 2.5})$$
Yüzdeye çevirmek için bu değeri 100 ile çarparız.

In [ ]:
def poisson_olasilik(k, lam):
    '''
    Poisson olasılık yoğunluk fonksiyonunu hesaplar.
    k: Hedeflenen gol sayısı (0, 1, 2, ...)
    lam (lambda): Beklenen ortalama gol beklentisi
    '''
    # (lam^k * e^-lam) / k!
    return (lam ** k) * math.exp(-lam) / math.factorial(k)

# Örnek Hesaplama:
# Bir maçta ev sahibinin gol beklentisi (lambda) 1.6 olsun.
print("Ortalama Gol Beklentisi 1.6 olan takımın:")
print(f"  - 0 Gol atma (golsüz kalma) olasılığı: %{poisson_olasilik(0, 1.6)*100:.2f}")
print(f"  - 1 Gol atma olasılığı:             %{poisson_olasilik(1, 1.6)*100:.2f}")
print(f"  - 2 Gol atma olasılığı:             %{poisson_olasilik(2, 1.6)*100:.2f}")
print(f"  - 3 Gol atma olasılığı:             %{poisson_olasilik(3, 1.6)*100:.2f}")

## 4. Bölüm: Algoritmanın Doğruluğunu Artıran İleri Düzey Katsayılar

Eğer sadece düz bir gol ortalaması alıp Poisson formülüne gönderirsek, genel performans grafiğini ve saha avantajını kaçıracağı için hatalı tahminler üretecektir. Algoritmanın isabet oranını artırmak için iki kritik katsayı geliştirdik:

### 4.1 Zaman Ağırlıklı Form Katsayısı (Form Decay)
Futbolda takımların en güncel maçlarındaki performansı, haftalar/aylar öncesine göre çok daha belirleyicidir. Örneğin son 3 maçında 4'er gol atan bir takım formda iken, 6 hafta önce gol rekoru kıran ama son 3 haftadır gol atamayan bir takım düşüştedir.

Bunu yansıtmak için **Üstel Azalım (Exponential Decay)** yöntemi uyguluyoruz.
En güncel maçtan (index 0) en eski maça (index $i$) doğru her maçın gol sayısını $w_i = (0.85)^i$ katsayısıyla ağırlıklandırırız:

* En son oynanan maç ($i=0$): $0.85^0 = 1.00$ (%100 etkili)
* 1 maç önceki maç ($i=1$): $0.85^1 = 0.85$ (%85 etkili)
* 2 maç önceki maç ($i=2$): $0.85^2 = 0.72$ (%72 etkili)
* 3 maç önceki maç ($i=3$): $0.85^3 = 0.61$ (%61 etkili)
* 4 maç önceki maç ($i=4$): $0.85^4 = 0.52$ (%52 etkili)

Ağırlıklı ortalamayı bulurken şu formülü çalıştırırız:
$$\text{Ağırlıklı Gol Ortalaması} = \frac{\sum (\text{Gol}_i \times w_i)}{\sum w_i}$$

### 4.2 Saha Avantajı (Home/Away Factor)
Futbolda taraftar desteği, saha alışkanlığı ve yolculuk yorgunluğu gibi sebeplerden dolayı ev sahibi takımlar genellikle daha avantajlıdır. Bu durum takımların hücum ve savunma istatistiklerine yansır.

* **Ev Sahibi (Takım 1) Gücü**: Takımın son $N$ maçındaki **genel performansına %40** ağırlık verirken, sadece **kendi evinde oynadığı** maçlardaki performansına **%60** ağırlık veririz.
* **Deplasman (Takım 2) Gücü**: Takımın son $N$ maçındaki **genel performansına %40** ağırlık verirken, sadece **deplasmanda oynadığı** maçlardaki performansına **%60** ağırlık veririz.

### 4.3 Kombine Edilmiş Beklenen Gol Değerleri ($\lambda_{ev}$ ve $\lambda_{dep}$)
Son aşamada takımların hücum gücü ile rakiplerinin savunma zafiyetini (yedikleri golleri) birleştirerek o maçtaki gol beklentilerini buluruz:

$$\lambda_{ev} = \frac{\text{Ev Sahibi Kombine Atan} + \text{Deplasman Kombine Yiyen}}{2}$$
$$\lambda_{dep} = \frac{\text{Deplasman Kombine Atan} + \text{Ev Sahibi Kombine Yiyen}}{2}$$

Bu sayede iki takımın birbirine karşı sergileyebileceği isabetli bir gol beklentisi elde edilir.

In [ ]:
def ust_2_5_olasilik_hesapla(takim1_mac_verileri, takim2_mac_verileri):
    '''
    İki takımın geçmiş maç verilerinden (kendi_gol, rakip_gol, is_home) hareketle;
    Form Decay katsayısı (0.85^i) ve Saha Avantajı (%60 iç/dış - %40 genel) uygulayarak
    beklenen gol parametrelerini (lambda) ve Alt/Üst 2.5 olasılıklarını hesaplar.
    '''
    def weighted_avg(veriler, filter_fn=None, get_rakip=False):
        '''Belirli bir saha filtresine göre (iç saha/deplasman) zaman ağırlıklı ortalama alır.'''
        total_val = 0.0
        total_weight = 0.0
        for i, (kendi, rakip, is_home) in enumerate(veriler):
            if filter_fn is None or filter_fn(is_home):
                # Form Decay Ağırlığı: Yeni maçlar daha büyük katsayı alır
                w = 0.85 ** i
                total_val += (rakip if get_rakip else kendi) * w
                total_weight += w
        return total_val / total_weight if total_weight > 0 else None

    # --- TAKIM 1 (Ev Sahibi) KOMBİNE DEĞERLERİ ---
    t1_genel_atan = weighted_avg(takim1_mac_verileri, get_rakip=False)
    t1_genel_yiyen = weighted_avg(takim1_mac_verileri, get_rakip=True)
    t1_ev_atan = weighted_avg(takim1_mac_verileri, filter_fn=lambda h: h is True, get_rakip=False)
    t1_ev_yiyen = weighted_avg(takim1_mac_verileri, filter_fn=lambda h: h is True, get_rakip=True)
    
    # Eğer takımın verilerinde hiç iç saha maçı yoksa genel ortalamayı kopyala
    if t1_ev_atan is None: t1_ev_atan = t1_genel_atan
    if t1_ev_yiyen is None: t1_ev_yiyen = t1_genel_yiyen
    
    # %40 Genel + %60 Ev Sahibi
    t1_combined_atan = 0.4 * t1_genel_atan + 0.6 * t1_ev_atan
    t1_combined_yiyen = 0.4 * t1_genel_yiyen + 0.6 * t1_ev_yiyen

    # --- TAKIM 2 (Deplasman) KOMBİNE DEĞERLERİ ---
    t2_genel_atan = weighted_avg(takim2_mac_verileri, get_rakip=False)
    t2_genel_yiyen = weighted_avg(takim2_mac_verileri, get_rakip=True)
    t2_dep_atan = weighted_avg(takim2_mac_verileri, filter_fn=lambda h: h is False, get_rakip=False)
    t2_dep_yiyen = weighted_avg(takim2_mac_verileri, filter_fn=lambda h: h is False, get_rakip=True)
    
    # Deplasman maçı yoksa genel ortalamayı kopyala
    if t2_dep_atan is None: t2_dep_atan = t2_genel_atan
    if t2_dep_yiyen is None: t2_dep_yiyen = t2_genel_yiyen
    
    # %40 Genel + %60 Deplasman
    t2_combined_atan = 0.4 * t2_genel_atan + 0.6 * t2_dep_atan
    t2_combined_yiyen = 0.4 * t2_genel_yiyen + 0.6 * t2_dep_yiyen

    # --- BEKLENEN MAÇ LAMBDALARI ---
    lambda_ev = (t1_combined_atan + t2_combined_yiyen) / 2
    lambda_dep = (t2_combined_atan + t1_combined_yiyen) / 2

    # --- ALT 2.5 HESAPLAMA (Toplam gol <= 2 olan skorlar) ---
    alt_2_5 = 0.0
    for ev_gol in range(3):      # 0, 1, 2 gol
        for dep_gol in range(3):  # 0, 1, 2 gol
            if ev_gol + dep_gol <= 2:
                # Bağımsız olasılıklar çarpılır
                alt_2_5 += poisson_olasilik(ev_gol, lambda_ev) * poisson_olasilik(dep_gol, lambda_dep)

    # Üst olasılığı = 1 - Alt olasılığı
    ust_2_5 = 1.0 - alt_2_5

    return lambda_ev, lambda_dep, ust_2_5

## 5. Bölüm: Yaklaşan Maçların Tarih Kontrolü ve Derbi Algoritması

Bahis analiz aracımız, sorgulanan iki takımın 1 hafta (7 gün) içerisinde birbirleriyle yapacakları bir derbi karşılaşmasının olup olmadığını kontrol eder. Eğer yaklaşan bir maç varsa, güncel istatistiklere göre maç tahmini yorumları hazırlar.

### 5.1 Türkçe Tarih Ayrıştırma Mantığı
Sporx fikstür sayfalarında yaklaşan maçların tarihi `"28 Mayıs 2026, 20:30"` gibi Türkçe metin formatında sunulur. Python'un tarih karşılaştırması yapabilmesi için bunu `datetime.date` nesnesine çevirmemiz gerekir:

1. **Metin Temizleme**: Virgüle göre metni böleriz. Virgülden öncesini alırız: `"28 Mayıs 2026"`.
2. **Parçalara Bölme**: Boşluk karakterine göre ayırarak Gün (`28`), Ay Adı (`"Mayıs"`) ve Yıl (`2026`) bilgilerini çekeriz.
3. **Ay Eşleme**: Türkçe ay adlarının takvim numaralarını tutan bir sözlük yardımıyla ay ismini sayıya çeviririz (Örn: `"mayıs"` -> `5`).
4. **Tarih Nesnesi Oluşturma**: `datetime.date(yil, ay, gun)` nesnesini tanımlarız.

### 5.2 Gün Farkı Ölçümü
Bugünün tarihini almak için `datetime.date.today()` metodunu çağırırız. Bulduğumuz maç tarihi ile bugünün tarihini çıkardığımızda Python bize `timedelta` nesnesi döner. Bu nesnenin `.days` özelliği iki tarih arasındaki gün farkını verir.

Eğer gün farkı `0 <= fark <= 7` ise, bu maçın 1 hafta içerisinde gerçekleşeceği anlaşılır ve ekranda alarm verilir.

In [ ]:
def turkce_tarih_ayristir(tarih_str):
    '''Türkçe tarih metnini datetime.date nesnesine dönüştürür.'''
    # Saat kısmını ayırmak için virgülden öncesini temizleriz
    tarih_kismi = tarih_str.split(",")[0].strip()
    
    # Günü, ay adını ve yılı boşlukla parçalarız
    parcalar = tarih_kismi.split(" ")
    if len(parcalar) >= 3:
        try:
            gun = int(parcalar[0])
            ay_adi = parcalar[1].lower()
            yil = int(parcalar[2])
            
            # Türkçe ayların numaraları
            aylar = {
                "ocak": 1, "subat": 2, "mart": 3, "nisan": 4, "mayis": 5, "haziran": 6,
                "temmuz": 7, "agustos": 8, "eylul": 9, "ekim": 10, "kasim": 11, "aralik": 12,
                "şubat": 2, "mayıs": 5, "ağustos": 8, "eylül": 9, "kasım": 11, "aralık": 12
            }
            ay = aylar.get(ay_adi, 1)
            return datetime.date(yil, ay, gun)
        except Exception:
            return None
    return None

# Algoritmayı test edelim
test_tarihi = "28 Mayıs 2026, 20:30"
parsed_date = turkce_tarih_ayristir(test_tarihi)
print("Ayrıştırılan Tarih Nesnesi:", parsed_date)
if parsed_date:
    today = datetime.date.today()
    diff = parsed_date - today
    print(f"Bugünün Tarihi: {today}")
    print(f"Maça Kalan Gün Sayısı: {diff.days} Gün")

## 6. Bölüm: HTML Veri Çekme Fonksiyonlarının Detaylı Analizi

Bu bölümde Sporx sitesinden verileri toplayan ana web kazıma (Web Scraping) fonksiyonlarını yazacağız.

### 6.1 `son_mac_bilgilerini_cek(takim, mac_sayisi)`
Bu fonksiyon, Poisson olasılık hesaplamasında ve son 3 maç istatistiğinde kullanılacak verileri çeker. 
* **reversed(maclar)**: Fikstür kronolojik sırayla gittiği için en yeni maç tablonun en altındadır. En yeni maçlardan eskiye doğru gitmek için listeyi tersten tararız.
* **Gol Bilgileri**: Skor etiketini tireye (`-`) göre bölüp ev sahibi ve deplasman gollerini alırız.
* **Saha Bilgisi**: Takımın ev sahibi mi yoksa deplasman mı olduğunu isme göre karşılaştırıp `True` veya `False` şeklinde listeye ekleriz. Sonuçta `(kendi_gol, rakip_gol, is_home)` demeti döner.

### 6.2 `takim_bilgilerini_cek(takim)`
Takımın sezon boyu oynadığı tüm maçları tarayarak genel galibiyet sayılarını, attığı toplam golleri ve en son maçının skor metnini hazırlar.

### 6.3 `gelecek_mac_kontrol(takim1, takim2)`
İki takımın fikstür sayfasındaki henüz oynanmamış (skoru `"-"` olan) ilk maçı bulur. Bu maçın tarihi 7 günden yakınsa derbi bilgisi döndürür.

In [ ]:
def son_mac_bilgilerini_cek(takim, mac_sayisi):
    '''
    Belirtilen takımın son N maçına ait gol verilerini çeker.
    Her maç için (kendi_gol, rakip_gol, is_home) formatında demet listesi döndürür.
    '''
    takim_seo = turkce_karakter_degistir(takim.lower())
    url = f"https://www.sporx.com/{takim_seo}-fiksturu-ve-mac-sonuclari"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")
    except Exception:
        return []

    maclar = soup.find_all("tr")
    mac_sonuclari = []
    mac_sayaci = 0

    # En güncel maçlardan eskiye gitmek için tersten döneriz
    for mac in reversed(maclar):
        skor_element = mac.find("a", class_="score")
        if skor_element:
            skor = skor_element.get_text(strip=True)
            gol_sayisi = skor.split("-")

            # Geçerli bir skor formatı mı kontrol edilir (örn: 2-1)
            if len(gol_sayisi) == 2 and gol_sayisi[0].strip() and gol_sayisi[1].strip():
                try:
                    ev_gol = int(gol_sayisi[0])
                    dep_gol = int(gol_sayisi[1])
                except ValueError:
                    continue

                ev_sahibi_div = mac.find("div", class_="team home")
                deplasman_div = mac.find("div", class_="team away")

                if ev_sahibi_div and deplasman_div:
                    ev_a = ev_sahibi_div.find("a", class_="team-name")
                    dep_a = deplasman_div.find("a", class_="team-name")

                    if ev_a and dep_a:
                        ev_isim = turkce_karakter_degistir(ev_a.get_text(strip=True).lower())
                        dep_isim = turkce_karakter_degistir(dep_a.get_text(strip=True).lower())

                        # Sorgulanan takım ev sahibi ise:
                        if takim.lower() == ev_isim:
                            mac_sonuclari.append((ev_gol, dep_gol, True)) # Kendi golü ev golüdür, is_home = True
                            mac_sayaci += 1
                        # Sorgulanan takım deplasman ise:
                        elif takim.lower() == dep_isim:
                            mac_sonuclari.append((dep_gol, ev_gol, False)) # Kendi golü deplasman golüdür, is_home = False
                            mac_sayaci += 1

                if mac_sayaci >= mac_sayisi:
                    break
    return mac_sonuclari

def takim_bilgilerini_cek(takim):
    '''
    Takımın tüm sezonluk verilerini tarayarak toplam galibiyetini,
    toplam gol sayısını ve en son oynadığı maç skorunu çeker.
    '''
    clear_screen()
    takim_seo = turkce_karakter_degistir(takim.lower())
    url = f"https://www.sporx.com/{takim_seo}-fiksturu-ve-mac-sonuclari"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")
    except Exception:
        return None

    maclar = soup.find_all("tr")
    galibiyet_sayisi = 0
    toplam_gol = 0
    son_mac_skoru = None

    for mac in maclar:
        skor_element = mac.find("a", class_="score")
        if skor_element:
            skor = skor_element.get_text(strip=True)
            gol_sayisi = skor.split("-")
            
            if len(gol_sayisi) == 2 and gol_sayisi[0].strip() and gol_sayisi[1].strip():
                try:
                    attigi_gol = int(gol_sayisi[0])
                    gol_sayisi_g2 = int(gol_sayisi[1])
                except ValueError:
                    continue
                
                ev_sahibi_div = mac.find("div", class_="team home")
                deplasman_div = mac.find("div", class_="team away")
                
                if ev_sahibi_div and deplasman_div:
                    ev_sahibi_a = ev_sahibi_div.find("a", class_="team-name")
                    deplasman_a = deplasman_div.find("a", class_="team-name")
                    
                    if ev_sahibi_a and deplasman_a:
                        ev_sahibi = ev_sahibi_a.get_text(strip=True)
                        deplasman = deplasman_a.get_text(strip=True)
                        
                        if takim.lower() == turkce_karakter_degistir(ev_sahibi.lower()):
                            toplam_gol += attigi_gol
                            if attigi_gol > gol_sayisi_g2:
                                galibiyet_sayisi += 1
                            son_mac_skoru = f"Son Maç: {ev_sahibi} {skor} {deplasman}\n"
                            
                        elif takim.lower() == turkce_karakter_degistir(deplasman.lower()):
                            toplam_gol += gol_sayisi_g2
                            if attigi_gol < gol_sayisi_g2:
                                galibiyet_sayisi += 1
                            son_mac_skoru = f"Son Maç: {ev_sahibi} {skor} {deplasman}\n"
                            
    if galibiyet_sayisi == 0:
        return None
    else:
        return galibiyet_sayisi, toplam_gol, son_mac_skoru

def gelecek_mac_kontrol(takim1, takim2):
    '''
    İki takımın 1 hafta içerisinde birbirleriyle yapacakları maçı arar.
    Maç bulursa detaylarını döndürür, yoksa None döner.
    '''
    takim1_seo = turkce_karakter_degistir(takim1.lower())
    url = f"https://www.sporx.com/{takim1_seo}-fiksturu-ve-mac-sonuclari"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")
    except Exception:
        return None

    maclar = soup.find_all("tr")
    today = datetime.date.today()

    for mac in maclar:
        skor_element = mac.find("a", class_="score")
        if skor_element:
            skor = skor_element.get_text(strip=True)
            if skor == "-":
                ev_sahibi_div = mac.find("div", class_="team home")
                deplasman_div = mac.find("div", class_="team away")
                if ev_sahibi_div and deplasman_div:
                    ev_a = ev_sahibi_div.find("a", class_="team-name")
                    dep_a = deplasman_div.find("a", class_="team-name")
                    if ev_a and dep_a:
                        ev_isim = turkce_karakter_degistir(ev_a.get_text(strip=True).lower())
                        dep_isim = turkce_karakter_degistir(dep_a.get_text(strip=True).lower())

                        t1_seo = turkce_karakter_degistir(takim1.lower())
                        t2_seo = turkce_karakter_degistir(takim2.lower())

                        # Karşılaşan takımlar bizim aradığımız iki takım mı?
                        if (t1_seo == ev_isim and t2_seo == dep_isim) or (t1_seo == dep_isim and t2_seo == ev_isim):
                            date_span = mac.find("span", class_="date")
                            if date_span:
                                tarih_str = date_span.get_text(strip=True)
                                mac_tarihi = turkce_tarih_ayristir(tarih_str)
                                if mac_tarihi:
                                    diff = mac_tarihi - today
                                    if 0 <= diff.days <= 7:
                                        return {
                                            "tarih": tarih_str,
                                            "ev_sahibi": ev_a.get_text(strip=True),
                                            "deplasman": dep_a.get_text(strip=True),
                                            "kalan_gun": diff.days
                                        }
    return None

## 7. Bölüm: Konsol Tabanlı Test Versiyonu ve Tetikleyici Mantığı

Arayüzün arkasındaki tetikleme mekanizmasını ve tüm fonksiyonların nasıl bir arada uyumla çalıştığını gözlemlemek için bir test fonksiyonu çalıştıracağız.

### 7.1 Çalışma Adımları
1. **Veri Çekme**: Ev sahibi ve deplasman takımlarının tüm fikstürü indirilerek genel galibiyet sayıları ve son oynadıkları maç skorları belirlenir.
2. **Form Durumu Belirleme**: Galibiyet sayıları kıyaslanarak form grafiğine göre **favori** veya **normal** olarak işaretlenir.
3. **Son N Maç Analizi**: Poisson için zaman katsayılı ($w = 0.85^i$) ve ev/deplasman avantajlı beklenen goller hesaplanıp Alt/Üst 2.5 olasılığı bulunur.
4. **Derbi Kontrolü**: Yakın zamanda (1 hafta içinde) maçları varsa sonuca eklenir, yoksa "Yakın zamanda karşılaşacakları maç yok" yazılır.

In [ ]:
def iki_takimli_analiz_konsol(takim1_girdi, takim2_girdi, mac_sayisi_girdi=7):
    '''
    Arayüz elemanları olmadan, algoritmanın ve veri çekme fonksiyonlarının
    nasıl çalıştığını konsol ekranında test etmemizi sağlayan fonksiyondur.
    '''
    takim1 = turkce_karakter_degistir(takim1_girdi)
    takim2 = turkce_karakter_degistir(takim2_girdi)
    mac_sayisi = int(mac_sayisi_girdi)
    
    print(f"🔄 {takim1.upper()} ve {takim2.upper()} analizi internetten çekilerek başlatılıyor... (Lütfen bekleyin)\n")
    
    # 1. Sezonluk genel verileri çekelim
    takim1_bilgileri = takim_bilgilerini_cek(takim1)
    takim2_bilgileri = takim_bilgilerini_cek(takim2)
    
    if takim1_bilgileri is None or takim2_bilgileri is None:
        print("❌ Takımların bilgileri internetten çekilemedi. İsimleri doğru yazdığınızdan emin olun.")
        return
    
    galibiyet_sayisi_g1, gol_sayisi_g1, son_mac_skoru_g1 = takim1_bilgileri
    galibiyet_sayisi_g2, gol_sayisi_g2, son_mac_skoru_g2 = takim2_bilgileri
    
    # Galibiyet sayısına göre form durumu (favori/normal)
    takim1_form = "favori" if galibiyet_sayisi_g1 > galibiyet_sayisi_g2 else "normal"
    takim2_form = "favori" if galibiyet_sayisi_g2 > galibiyet_sayisi_g1 else "normal"
    
    # 2. Son 3 maçın detaylarını çekelim
    takim1_son_3_mac = son_mac_bilgilerini_cek(takim1, 3)
    takim2_son_3_mac = son_mac_bilgilerini_cek(takim2, 3)
    
    if len(takim1_son_3_mac) < 3 or len(takim2_son_3_mac) < 3:
        print("❌ Son 3 maç verisi bulunamadı.")
        return
    
    # Son 3 maçta galibiyet/beraberlik/malubiyet sayıları
    t1_kazanma = sum(1 for kendi, rakip, *is_home in takim1_son_3_mac if kendi > rakip)
    t1_beraberlik = sum(1 for kendi, rakip, *is_home in takim1_son_3_mac if kendi == rakip)
    t1_malubiyet = sum(1 for kendi, rakip, *is_home in takim1_son_3_mac if kendi < rakip)
    
    t2_kazanma = sum(1 for kendi, rakip, *is_home in takim2_son_3_mac if kendi > rakip)
    t2_beraberlik = sum(1 for kendi, rakip, *is_home in takim2_son_3_mac if kendi == rakip)
    t2_malubiyet = sum(1 for kendi, rakip, *is_home in takim2_son_3_mac if kendi < rakip)
    
    # Son 3 maçta attıkları toplam gol
    t1_toplam_gol = sum(kendi for kendi, rakip, *is_home in takim1_son_3_mac)
    t2_toplam_gol = sum(kendi for kendi, rakip, *is_home in takim2_son_3_mac)
    
    # 3. Poisson Analizi için gerekli son N maçın gollerini çekelim
    takim1_tum_mac_gol = son_mac_bilgilerini_cek(takim1, mac_sayisi)
    takim2_tum_mac_gol = son_mac_bilgilerini_cek(takim2, mac_sayisi)
    
    if len(takim1_tum_mac_gol) < mac_sayisi or len(takim2_tum_mac_gol) < mac_sayisi:
        print("❌ Gol tahmini için yeterli geçmiş maç bulunamadı.")
        return
    
    # Poisson Olasılıklarını Hesapla
    lambda_ev, lambda_dep, ust_2_5 = ust_2_5_olasilik_hesapla(takim1_tum_mac_gol, takim2_tum_mac_gol)
    
    # Ortalama goller
    t1_ort = sum(k for k, r, *is_home in takim1_tum_mac_gol) / len(takim1_tum_mac_gol)
    t2_ort = sum(k for k, r, *is_home in takim2_tum_mac_gol) / len(takim2_tum_mac_gol)
    
    # Sonuçları ekrana yazdıralım
    print("==================================================")
    print(f"📊 {takim1_girdi.upper()} vs {takim2_girdi.upper()} ANALİZ RAPORU")
    print("==================================================")
    print(f"{takim1_girdi.capitalize()} Form Durumu: {takim1_form.upper()}")
    print(f"{takim2_girdi.capitalize()} Form Durumu: {takim2_form.upper()}\n")
    
    print(f"⚽ {takim1_girdi.capitalize()} Son 3 Maç İstatistiği:")
    print(f"   Kazanma: {t1_kazanma} | Beraberlik: {t1_beraberlik} | Mağlubiyet: {t1_malubiyet}")
    print(f"   Atılan Gol Sayısı (Son 3 Maç): {t1_toplam_gol}\n")
    
    print(f"⚽ {takim2_girdi.capitalize()} Son 3 Maç İstatistiği:")
    print(f"   Kazanma: {t2_kazanma} | Beraberlik: {t2_beraberlik} | Mağlubiyet: {t2_malubiyet}")
    print(f"   Atılan Gol Sayısı (Son 3 Maç): {t2_toplam_gol}\n")
    
    print(f"📈 Maç Başına Ortalama Gol (Son {mac_sayisi} Maç):")
    print(f"   {takim1_girdi.capitalize()}: {t1_ort:.2f} gol/maç")
    print(f"   {takim2_girdi.capitalize()}: {t2_ort:.2f} gol/maç\n")
    
    print(f"🧮 Gelişmiş Poisson Gol Beklentisi (Lambda):")
    print(f"   {takim1_girdi.capitalize()} (Ev): {lambda_ev:.2f} gol")
    print(f"   {takim2_girdi.capitalize()} (Dep): {lambda_dep:.2f} gol\n")
    
    ust_yuzde = ust_2_5 * 100
    print(f"🎯 Olasılık Tahmin Oranları:")
    print(f"   Maçın Üst 2.5 Gol Bitme Olasılığı: %{ust_yuzde:.1f}")
    print(f"   Maçın Alt 2.5 Gol Bitme Olasılığı: %{(100 - ust_yuzde):.1f}\n")
    
    # Gelecek Maç Kontrolü
    gelecek_mac = gelecek_mac_kontrol(takim1, takim2)
    if gelecek_mac:
        print(f"📣 YAKLAŞAN MAÇ BİLGİSİ ({gelecek_mac['tarih']}):")
        print(f"   Bu iki takım 1 hafta içinde karşı karşıya gelecektir!")
        print(f"   Kalan Gün: {gelecek_mac['kalan_gun']} Gün")
        if lambda_ev > lambda_dep + 0.3:
            print(f"   Yorum: Ev sahibi {gelecek_mac['ev_sahibi']} galibiyete daha yakın görünüyor.")
        elif lambda_dep > lambda_ev + 0.3:
            print(f"   Yorum: Deplasman {gelecek_mac['deplasman']} galibiyete daha yakın görünüyor.")
        else:
            print(f"   Yorum: Mücadele oldukça dengeli, beraberlik olasılığı yüksek.")
    else:
        print("📣 YAKLAŞAN MAÇ BİLGİSİ:")
        print("   Bu iki takımın yakın zamanda (1 hafta içerisinde) karşılaşacağı bir maç bulunmuyor.")

# Deneme testi çalıştır (İnternet bağlantısı gereklidir)
# Gerçek takım isimleri girerek veriyi Sporx'ten çekelim
iki_takimli_analiz_konsol("Galatasaray", "Fenerbahçe", 7)

## 8. Bölüm: Grafiksel Kullanıcı Arayüzü (Tkinter GUI) ve Tam Kod

Aşağıdaki hücrede, projenin tüm mantıksal katmanlarını ve Tkinter GUI kodunu bir araya getiren **tam çalıştırılabilir kod bloğu** yer almaktadır.

### 8.1 Tkinter Yapısı ve Event Loop (Olay Döngüsü)
Tkinter uygulamaları **olay güdümlüdür (event-driven)**. `root.mainloop()` komutu çalıştırıldığında program sonsuz bir döngüye girer. Bu döngü, kullanıcının butona tıklaması, metin girmesi veya pencereyi kapatması gibi olayları saniyede binlerce kez tarayarak bekler. Butona tıklandığında ilişkili fonksiyon (`command=iki_takimli_analiz`) çalıştırılır.

### 8.2 Grid (Izgara) Düzen Yöneticisi
Arayüzün düzgün hizalanması için `grid` yerleşim sistemini kullanırız:
* `row` ve `column`: Hücre koordinatlarıdır (0,0 sol üst köşedir).
* `sticky="w"`: Widget'ı hücresinin sol kenarına (West/Batı) yaslar.
* `columnspan=2`: Widget'ın 2 sütun genişliğinde yer kaplamasını sağlar.
* `padx` ve `pady`: Piksel cinsinden dış boşluklar ekler.

### 8.3 Dinamik Pencere Boyutlandırma
Analiz bittikten sonra sonuç metninin sığması için `root.geometry("")` komutu verilir. Boş metin verilmesi, Tkinter'ın pencere boyutunu içerideki yazının ve nesnelerin büyüklüğüne göre otomatik genişletmesini sağlar.

> ⚠️ **ÖNEMLİ UYARI:** Aşağıdaki kodu çalıştırdığınızda bilgisayarınızda görsel bir masaüstü penceresi açılacaktır. Uygulamayı pencere köşesindeki **[X]** (kapat) butonuyla kapatana kadar bu hücre meşgul durumda kalacaktır. Hücre çalışırken solundaki `[*]` işareti meşgul olduğunu gösterir. Pencereyi kapattığınızda hücrenin çalışması tamamlanacaktır.

In [ ]:
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import requests
from bs4 import BeautifulSoup
import os
import math
import datetime

DEFAULT_MAC_SAYISI = 7

def clear_screen():
    '''Konsol ekranını temizler.'''
    os.system('cls' if os.name == 'nt' else 'clear')

def turkce_karakter_degistir(takim_ad):
    '''Türkçe karakterleri güvenli bir şekilde ingilizce karakterlerle değiştirir.'''
    takim_ad = takim_ad.replace("İ", "i").replace("I", "i")
    takim_ad = takim_ad.lower()
    takim_ad = takim_ad.replace("ı", "i")
    takim_ad = takim_ad.replace("ç", "c")
    takim_ad = takim_ad.replace("ş", "s")
    takim_ad = takim_ad.replace("ğ", "g")
    takim_ad = takim_ad.replace("ü", "u")
    takim_ad = takim_ad.replace("ö", "o")
    return takim_ad.replace(" ", "-")

def turkce_tarih_ayristir(tarih_str):
    '''Sporx'ten gelen Türkçe tarih string metnini datetime.date nesnesine çevirir.'''
    tarih_kismi = tarih_str.split(",")[0].strip()
    parcalar = tarih_kismi.split(" ")
    if len(parcalar) >= 3:
        try:
            gun = int(parcalar[0])
            ay_adi = parcalar[1].lower()
            yil = int(parcalar[2])
            aylar = {
                "ocak": 1, "subat": 2, "mart": 3, "nisan": 4, "mayis": 5, "haziran": 6,
                "temmuz": 7, "agustos": 8, "eylul": 9, "ekim": 10, "kasim": 11, "aralik": 12,
                "şubat": 2, "mayıs": 5, "ağustos": 8, "eylül": 9, "kasım": 11, "aralık": 12
            }
            ay = aylar.get(ay_adi, 1)
            return datetime.date(yil, ay, gun)
        except Exception:
            return None
    return None

def poisson_olasilik(k, lam):
    '''Poisson olasılık formülünü uygular.'''
    return (lam ** k) * math.exp(-lam) / math.factorial(k)

def ust_2_5_olasilik_hesapla(takim1_mac_verileri, takim2_mac_verileri):
    '''Zaman ve saha ağırlıklı Poisson hesabı yapar.'''
    def weighted_avg(veriler, filter_fn=None, get_rakip=False):
        total_val = 0.0
        total_weight = 0.0
        for i, (kendi, rakip, is_home) in enumerate(veriler):
            if filter_fn is None or filter_fn(is_home):
                w = 0.85 ** i
                total_val += (rakip if get_rakip else kendi) * w
                total_weight += w
        return total_val / total_weight if total_weight > 0 else None

    # Ev sahibi
    t1_genel_atan = weighted_avg(takim1_mac_verileri, get_rakip=False)
    t1_genel_yiyen = weighted_avg(takim1_mac_verileri, get_rakip=True)
    t1_ev_atan = weighted_avg(takim1_mac_verileri, filter_fn=lambda h: h is True, get_rakip=False)
    t1_ev_yiyen = weighted_avg(takim1_mac_verileri, filter_fn=lambda h: h is True, get_rakip=True)
    if t1_ev_atan is None: t1_ev_atan = t1_genel_atan
    if t1_ev_yiyen is None: t1_ev_yiyen = t1_genel_yiyen
    t1_combined_atan = 0.4 * t1_genel_atan + 0.6 * t1_ev_atan
    t1_combined_yiyen = 0.4 * t1_genel_yiyen + 0.6 * t1_ev_yiyen

    # Deplasman
    t2_genel_atan = weighted_avg(takim2_mac_verileri, get_rakip=False)
    t2_genel_yiyen = weighted_avg(takim2_mac_verileri, get_rakip=True)
    t2_dep_atan = weighted_avg(takim2_mac_verileri, filter_fn=lambda h: h is False, get_rakip=False)
    t2_dep_yiyen = weighted_avg(takim2_mac_verileri, filter_fn=lambda h: h is False, get_rakip=True)
    if t2_dep_atan is None: t2_dep_atan = t2_genel_atan
    if t2_dep_yiyen is None: t2_dep_yiyen = t2_genel_yiyen
    t2_combined_atan = 0.4 * t2_genel_atan + 0.6 * t2_dep_atan
    t2_combined_yiyen = 0.4 * t2_genel_yiyen + 0.6 * t2_dep_yiyen

    # Lambda
    lambda_ev = (t1_combined_atan + t2_combined_yiyen) / 2
    lambda_dep = (t2_combined_atan + t1_combined_yiyen) / 2

    alt_2_5 = 0.0
    for ev_gol in range(3):
        for dep_gol in range(3):
            if ev_gol + dep_gol <= 2:
                alt_2_5 += poisson_olasilik(ev_gol, lambda_ev) * poisson_olasilik(dep_gol, lambda_dep)

    ust_2_5 = 1.0 - alt_2_5
    return lambda_ev, lambda_dep, ust_2_5

def takim_bilgilerini_cek(takim):
    '''Sezonluk gol ve galibiyet bilgilerini Sporx'ten çeker.'''
    clear_screen()
    takim_seo = turkce_karakter_degistir(takim.lower())
    url = f"https://www.sporx.com/{takim_seo}-fiksturu-ve-mac-sonuclari"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")
    except Exception:
        return None
    maclar = soup.find_all("tr")
    galibiyet_sayisi = 0
    toplam_gol = 0
    son_mac_skoru = None
    for mac in maclar:
        skor_element = mac.find("a", class_="score")
        if skor_element:
            skor = skor_element.get_text(strip=True)
            gol_sayisi = skor.split("-")
            if len(gol_sayisi) == 2 and gol_sayisi[0].strip() and gol_sayisi[1].strip():
                try:
                    attigi_gol = int(gol_sayisi[0])
                    gol_sayisi_g2 = int(gol_sayisi[1])
                except ValueError:
                    continue
                ev_sahibi_div = mac.find("div", class_="team home")
                deplasman_div = mac.find("div", class_="team away")
                if ev_sahibi_div and deplasman_div:
                    ev_sahibi_a = ev_sahibi_div.find("a", class_="team-name")
                    deplasman_a = deplasman_div.find("a", class_="team-name")
                    if ev_sahibi_a and deplasman_a:
                        ev_sahibi = ev_sahibi_a.get_text(strip=True)
                        deplasman = deplasman_a.get_text(strip=True)
                        if takim.lower() == turkce_karakter_degistir(ev_sahibi.lower()):
                            toplam_gol += attigi_gol
                            if attigi_gol > gol_sayisi_g2:
                                galibiyet_sayisi += 1
                            son_mac_skoru = f"Son Maç: {ev_sahibi} {skor} {deplasman}\n"
                        elif takim.lower() == turkce_karakter_degistir(deplasman.lower()):
                            toplam_gol += gol_sayisi_g2
                            if attigi_gol < gol_sayisi_g2:
                                galibiyet_sayisi += 1
                            son_mac_skoru = f"Son Maç: {ev_sahibi} {skor} {deplasman}\n"
    if galibiyet_sayisi == 0:
        return None
    return galibiyet_sayisi, toplam_gol, son_mac_skoru

def son_mac_bilgilerini_cek(takim, mac_sayisi):
    '''Son N maçın skorunu reversed döngüyle hiyerarşik HTML'den çeker.'''
    takim_seo = turkce_karakter_degistir(takim.lower())
    url = f"https://www.sporx.com/{takim_seo}-fiksturu-ve-mac-sonuclari"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")
    except Exception:
        return []
    maclar = soup.find_all("tr")
    mac_sonuclari = []
    mac_sayaci = 0
    for mac in reversed(maclar):
        skor_element = mac.find("a", class_="score")
        if skor_element:
            skor = skor_element.get_text(strip=True)
            gol_sayisi = skor.split("-")
            if len(gol_sayisi) == 2 and gol_sayisi[0].strip() and gol_sayisi[1].strip():
                try:
                    ev_gol = int(gol_sayisi[0])
                    dep_gol = int(gol_sayisi[1])
                except ValueError:
                    continue
                ev_sahibi_div = mac.find("div", class_="team home")
                deplasman_div = mac.find("div", class_="team away")
                if ev_sahibi_div and deplasman_div:
                    ev_a = ev_sahibi_div.find("a", class_="team-name")
                    dep_a = deplasman_div.find("a", class_="team-name")
                    if ev_a and dep_a:
                        ev_isim = turkce_karakter_degistir(ev_a.get_text(strip=True).lower())
                        dep_isim = turkce_karakter_degistir(dep_a.get_text(strip=True).lower())
                        if takim.lower() == ev_isim:
                            mac_sonuclari.append((ev_gol, dep_gol, True))
                            mac_sayaci += 1
                        elif takim.lower() == dep_isim:
                            mac_sonuclari.append((dep_gol, ev_gol, False))
                            mac_sayaci += 1
                if mac_sayaci >= mac_sayisi:
                    break
    return mac_sonuclari

def gelecek_mac_kontrol(takim1, takim2):
    '''Yaklaşan derbi olup olmadığını kontrol eder.'''
    takim1_seo = turkce_karakter_degistir(takim1.lower())
    url = f"https://www.sporx.com/{takim1_seo}-fiksturu-ve-mac-sonuclari"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")
    except Exception:
        return None
    maclar = soup.find_all("tr")
    today = datetime.date.today()
    for mac in maclar:
        skor_element = mac.find("a", class_="score")
        if skor_element:
            skor = skor_element.get_text(strip=True)
            if skor == "-":
                ev_sahibi_div = mac.find("div", class_="team home")
                deplasman_div = mac.find("div", class_="team away")
                if ev_sahibi_div and deplasman_div:
                    ev_a = ev_sahibi_div.find("a", class_="team-name")
                    dep_a = deplasman_div.find("a", class_="team-name")
                    if ev_a and dep_a:
                        ev_isim = turkce_karakter_degistir(ev_a.get_text(strip=True).lower())
                        dep_isim = turkce_karakter_degistir(dep_a.get_text(strip=True).lower())
                        t1_seo = turkce_karakter_degistir(takim1.lower())
                        t2_seo = turkce_karakter_degistir(takim2.lower())
                        if (t1_seo == ev_isim and t2_seo == dep_isim) or (t1_seo == dep_isim and t2_seo == ev_isim):
                            date_span = mac.find("span", class_="date")
                            if date_span:
                                tarih_str = date_span.get_text(strip=True)
                                mac_tarihi = turkce_tarih_ayristir(tarih_str)
                                if mac_tarihi:
                                    diff = mac_tarihi - today
                                    if 0 <= diff.days <= 7:
                                        return {
                                            "tarih": tarih_str,
                                            "ev_sahibi": ev_a.get_text(strip=True),
                                            "deplasman": dep_a.get_text(strip=True),
                                            "kalan_gun": diff.days
                                        }
    return None

def iki_takimli_analiz():
    '''Arayüz butonunun tetiklediği ana karar mekanizmasıdır.'''
    takim1 = turkce_karakter_degistir(takim1_entry.get())
    takim2 = turkce_karakter_degistir(takim2_entry.get())
    mac_sayisi = int(mac_sayisi_entry.get())
    
    if not takim1 or not takim2:
        messagebox.showerror("Hata", "Lütfen takımları girin.")
        return
    
    takim1_bilgileri = takim_bilgilerini_cek(takim1)
    takim2_bilgileri = takim_bilgilerini_cek(takim2)
    
    if takim1_bilgileri is None or takim2_bilgileri is None:
        return
    
    galibiyet_sayisi_g1, gol_sayisi_g1, son_mac_skoru_g1 = takim1_bilgileri
    galibiyet_sayisi_g2, gol_sayisi_g2, son_mac_skoru_g2 = takim2_bilgileri
    
    takim1_form = "favori" if galibiyet_sayisi_g1 > galibiyet_sayisi_g2 else "normal"
    takim2_form = "favori" if galibiyet_sayisi_g2 > galibiyet_sayisi_g1 else "normal"
    
    takim1_son_3_mac = son_mac_bilgilerini_cek(takim1, 3)
    takim2_son_3_mac = son_mac_bilgilerini_cek(takim2, 3)
    
    if len(takim1_son_3_mac) < 3 or len(takim2_son_3_mac) < 3:
        messagebox.showerror("Hata", "Son 3 maç verisi bulunamadı.")
        return
    
    # Son 3 maç galibiyet/beraberlik/malubiyet sayıları
    takim1_kazanma_sayisi = sum(1 for kendi, rakip, *is_home in takim1_son_3_mac if kendi > rakip)
    takim1_beraberlik_sayisi = sum(1 for kendi, rakip, *is_home in takim1_son_3_mac if kendi == rakip)
    takim1_malubiyet_sayisi = sum(1 for kendi, rakip, *is_home in takim1_son_3_mac if kendi < rakip)
    
    takim2_kazanma_sayisi = sum(1 for kendi, rakip, *is_home in takim2_son_3_mac if kendi > rakip)
    takim2_beraberlik_sayisi = sum(1 for kendi, rakip, *is_home in takim2_son_3_mac if kendi == rakip)
    takim2_malubiyet_sayisi = sum(1 for kendi, rakip, *is_home in takim2_son_3_mac if kendi < rakip)
    
    # Son N maç için tüm gol verileri
    takim1_tum_mac_gol = son_mac_bilgilerini_cek(takim1, mac_sayisi)
    takim2_tum_mac_gol = son_mac_bilgilerini_cek(takim2, mac_sayisi)
    
    if len(takim1_tum_mac_gol) < mac_sayisi or len(takim2_tum_mac_gol) < mac_sayisi:
        messagebox.showerror("Hata", "Gol tahmini yapmak için yeterli veri bulunamadı.")
        return
    
    # Son 3 maçta attığı goller
    takim1_toplam_gol = sum(kendi for kendi, rakip, *is_home in takim1_son_3_mac)
    takim2_toplam_gol = sum(kendi for kendi, rakip, *is_home in takim2_son_3_mac)
    
    lambda_ev, lambda_dep, ust_2_5 = ust_2_5_olasilik_hesapla(takim1_tum_mac_gol, takim2_tum_mac_gol)
    
    t1_ort = sum(k for k, r, *is_home in takim1_tum_mac_gol) / len(takim1_tum_mac_gol)
    t2_ort = sum(k for k, r, *is_home in takim2_tum_mac_gol) / len(takim2_tum_mac_gol)
    
    sonuc = f"{takim1.capitalize()} Takımı Form Durumu: {takim1_form}\n"
    sonuc += f"{takim2.capitalize()} Takımı Form Durumu: {takim2_form}\n\n"
    sonuc += f"{takim1.capitalize()} Son 3 Maçta:\nKazanma Sayısı: {takim1_kazanma_sayisi}\nBeraberlik Sayısı: {takim1_beraberlik_sayisi}\nMalubiyet Sayısı: {takim1_malubiyet_sayisi}\nAttığı Gol Sayısı: {takim1_toplam_gol}\n\n"
    sonuc += f"{takim2.capitalize()} Son 3 Maçta:\nKazanma Sayısı: {takim2_kazanma_sayisi}\nBeraberlik Sayısı: {takim2_beraberlik_sayisi}\nMalubiyet Sayısı: {takim2_malubiyet_sayisi}\nAttığı Gol Sayısı: {takim2_toplam_gol}\n\n"
    sonuc += f"{takim1.capitalize()} Maç Başına Ort. Gol: {t1_ort:.2f}\n"
    sonuc += f"{takim2.capitalize()} Maç Başına Ort. Gol: {t2_ort:.2f}\n\n"
    sonuc += f"Beklenen Gol (Poisson): {takim1.capitalize()} {lambda_ev:.2f} - {lambda_dep:.2f} {takim2.capitalize()}\n"
    sonuc += "(Poisson: Takımların gol atma ve yeme ortalamalarını analiz ederek, bu maçta atabilecekleri tahmini gol sayılarını hesaplayan matematiksel modeldir.)\n\n"
    
    ust_yuzde = ust_2_5 * 100
    if ust_yuzde >= 70:
        sonuc += f"Maçın Üst 2.5 Bitme Olasılığı: %{ust_yuzde:.1f} (Çok Yüksek)\n"
    elif ust_yuzde >= 50:
        sonuc += f"Maçın Üst 2.5 Bitme Olasılığı: %{ust_yuzde:.1f} (Yüksek)\n"
    elif ust_yuzde >= 35:
        sonuc += f"Maçın Üst 2.5 Bitme Olasılığı: %{ust_yuzde:.1f} (Orta)\n"
    else:
        sonuc += f"Maçın Üst 2.5 Bitme Olasılığı: %{ust_yuzde:.1f} (Düşük)\n"
    sonuc += f"Maçın Alt 2.5 Bitme Olasılığı: %{(100 - ust_yuzde):.1f}\n"
    
    gelecek_mac = gelecek_mac_kontrol(takim1, takim2)
    if gelecek_mac:
        tarih_str = gelecek_mac["tarih"]
        ev = gelecek_mac["ev_sahibi"]
        dep = gelecek_mac["deplasman"]
        
        if lambda_ev > lambda_dep + 0.3:
            tahmin_yorum = f"Ev sahibi {ev} ekibi galibiyete daha yakın görünüyor."
        elif lambda_dep > lambda_ev + 0.3:
            tahmin_yorum = f"Deplasman {dep} ekibi galibiyete daha yakın görünüyor."
        else:
            tahmin_yorum = "Mücadele oldukça dengeli, beraberlik olasılığı yüksek görünüyor."
            
        if ust_yuzde >= 50:
            gol_tahmin_yorum = f"maçın Üst 2.5 Gol (%{ust_yuzde:.1f}) bitme ihtimali daha yüksektir."
        else:
            gol_tahmin_yorum = f"maçın Alt 2.5 Gol (%{100 - ust_yuzde:.1f}) bitme ihtimali daha yüksektir."
            
        sonuc += f"\n📣 YAKLAŞAN MAÇ BİLGİSİ ({tarih_str}):\n"
        sonuc += f"Bu iki takımın 1 hafta içerisinde karşı karşıya geleceği bir maçı bulunuyor.\n"
        sonuc += f"Mevcut oranlara göre: {tahmin_yorum}\n"
        sonuc += f"Ayrıca {gol_tahmin_yorum}\n"
    else:
        sonuc += f"\n📣 YAKLAŞAN MAÇ BİLGİSİ:\n"
        sonuc += f"Bu iki takımın yakın zamanda (1 hafta içerisinde) karşılaşacağı bir maç bulunmuyor.\n"
        
    sonuc_label.config(text=sonuc)
    root.geometry("")

# --- Tkinter Arayüz Kurulumu ---
root = tk.Tk()
root.title("Futbol Analiz Programı")
root.resizable(False, False)

frame = ttk.Frame(root)
frame.grid(row=0, column=0, padx=10, pady=10)

takim1_label = ttk.Label(frame, text="Ev Sahibi Takım:")
takim1_label.grid(row=0, column=0, sticky="w")

takim1_entry = ttk.Entry(frame)
takim1_entry.grid(row=0, column=1, padx=5, pady=5)

takim2_label = ttk.Label(frame, text="Deplasman Takım:")
takim2_label.grid(row=1, column=0, sticky="w")

takim2_entry = ttk.Entry(frame)
takim2_entry.grid(row=1, column=1, padx=5, pady=5)

mac_sayisi_label = ttk.Label(frame, text="Gol analizi için kullanılacak son maç sayısı:")
mac_sayisi_label.grid(row=2, column=0, sticky="w")

mac_sayisi_entry = ttk.Entry(frame)
mac_sayisi_entry.insert(0, str(DEFAULT_MAC_SAYISI))
mac_sayisi_entry.grid(row=2, column=1, padx=5, pady=5)

analiz_button = ttk.Button(frame, text="Analiz Yap", command=iki_takimli_analiz)
analiz_button.grid(row=3, column=0, columnspan=2, pady=10)

sonuc_label = ttk.Label(frame, text="", justify="left", anchor="w")
sonuc_label.grid(row=4, column=0, columnspan=2, sticky="w", pady=10)

# Arayüzü çalıştıralım
# NOT: Çalıştırdığınızda harici bir Tkinter penceresi açılacaktır.
root.mainloop()